# Week 9 — CVaR Efficient Frontier

The key distributional result: plots mean PnL vs CVaR₀.₁₀ across α values.

**α-sweep:** `α ∈ {0.05, 0.10, 0.25, 0.50, 1.0}`  
Agent: QR-DQN + autoencoder, high_vol regime

**Expected shape:** non-dominated frontier where lower α → better tail protection
at cost of lower mean PnL. If some α dominates on both axes — report honestly.

**Sanity check:** at α=1.0 the policy is risk-neutral and should behave like DQN.
As α decreases, spreads should widen and MAP should decrease.

**Inputs:** `logs/qrdqn_autoencoder_asymmetric_high_vol_alpha*_seed42/`  
**Outputs:** `experiments/w09_frontier/`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from evaluation.efficient_frontier import EfficientFrontier, DEFAULT_ALPHAS
from evaluation.visualize import Visualizer, THEME, _dark_fig, _label, _legend
from training.evaluate import load_agent, evaluate_checkpoint
from envs.lob_env import LOBMarketMakingEnv

LOG_ROOT  = Path('../logs')
CKPT_ROOT = Path('../checkpoints')
EXP_DIR   = Path('../experiments/w09_frontier')
EXP_DIR.mkdir(parents=True, exist_ok=True)

AGENT   = 'qrdqn'
ENCODER = 'autoencoder'
REGIME  = 'high_vol'
REWARD  = 'asymmetric'
SEED    = 42
ALPHAS  = [0.05, 0.10, 0.25, 0.50, 1.0]

ef  = EfficientFrontier(
    log_root = LOG_ROOT,
    agent    = AGENT,
    encoder  = ENCODER,
    regime   = REGIME,
    seed     = SEED,
    alphas   = ALPHAS,
)
viz = Visualizer(log_root=LOG_ROOT, ckpt_root=CKPT_ROOT, out_root=EXP_DIR)
print('Setup complete')

## 1 — Build frontier DataFrame

In [ ]:
frontier_df = ef.build_frontier(episodes_window=50)

if frontier_df.empty:
    print('No frontier data found.')
    print('Run CVaR α-sweep first:')
    print('  python training/train.py agent=qrdqn encoder=autoencoder')
    print('    reward=asymmetric env.regime=high_vol seed=42')
    print('    alpha=0.05,0.10,0.25,0.50,1.0 --multirun')
else:
    display(frontier_df.style.format('{:.4f}', subset=[
        c for c in frontier_df.columns if frontier_df[c].dtype == float
    ]))
    frontier_df.to_csv(EXP_DIR / 'frontier_data.csv', index=False)
    print(f'Saved → {EXP_DIR}/frontier_data.csv')

## 2 — Pareto dominance analysis

In [ ]:
if not frontier_df.empty:
    pareto_df = ef.pareto_analysis(frontier_df)
    display(pareto_df[['alpha', 'mean_pnl', 'cvar_10', 'dominated']])

    dominated   = pareto_df[pareto_df['dominated']]
    undominated = pareto_df[~pareto_df['dominated']]

    print(f'\nNon-dominated points: {len(undominated)}')
    print(f'Dominated points:     {len(dominated)}')

    if dominated.empty:
        print('\nAll α values are non-dominated — clean efficient frontier.')
    else:
        print(f'\nDominated α values: {dominated["alpha"].tolist()}')
        print('Some α values are dominated — one α achieves both better')
        print('mean PnL AND better CVaR. Report this honestly in write-up:')
        print('the risk-averse policy found a genuinely better strategy.')

## 3 — Sanity checks

In [ ]:
checks = ef.sanity_checks(frontier_df)
print('Sanity checks:')
print('─' * 55)
for k, v in checks['details'].items():
    status = checks.get(f'{k}_pass')
    flag   = '✓ PASS' if status else ('✗ FAIL' if status is False else '? N/A')
    print(f'  {flag}  {v}')

print()
if checks.get('check1_pass') is False:
    print('WARNING: α=1.0 does not have highest mean PnL.')
    print('Check cvar_policy.py — at α=1.0 CVaR = mean, policy should be risk-neutral.')

if checks.get('check2_pass') is False:
    print('WARNING: MAP does not decrease with lower α.')
    print('Risk-averse agent should reduce inventory faster. Check reward_type and eta.')

if checks.get('check3_pass') is False:
    print('WARNING: Tail protection does not improve with lower α.')
    print('CVaR action selection may not be differentiating from mean policy.')
    print('Run CVaR vs mean divergence check in notebook 05.')

## 4 — Figure 3: Efficient frontier plot

In [ ]:
viz.plot_efficient_frontier(
    alphas  = ALPHAS,
    agent   = AGENT,
    encoder = ENCODER,
    regime  = REGIME,
    seed    = SEED,
    save    = True,
)

## 5 — Extended frontier: Sharpe and MAP by α

In [ ]:
if not frontier_df.empty:
    metrics_to_plot = [
        ('mean_pnl',  'Mean Episode PnL',    False),
        ('cvar_10',   'CVaR₀.₁₀',            False),
        ('sharpe',    'Sharpe Ratio',         False),
        ('map',       'MAP (lower = better)', True),
    ]
    metrics_to_plot = [
        (m, l, inv) for m, l, inv in metrics_to_plot if m in frontier_df.columns
    ]

    if metrics_to_plot:
        fig, axes = _dark_fig(
            figsize=(5 * len(metrics_to_plot), 4),
            nrows=1, ncols=len(metrics_to_plot)
        )
        if len(metrics_to_plot) == 1:
            axes = np.array([[axes]])

        alphas_available = frontier_df['alpha'].values

        for ax, (metric, label, invert) in zip(axes.flat, metrics_to_plot):
            vals = frontier_df[metric].values
            ax.plot(alphas_available, vals,
                    color='#f59e0b', lw=2, marker='o', ms=6)
            for a, v in zip(alphas_available, vals):
                ax.annotate(f'{v:.3f}',
                            (a, v), textcoords='offset points',
                            xytext=(0, 8), ha='center',
                            color=THEME['subtext'], fontsize=8)
            ax.axhline(0, color=THEME['subtext'], lw=0.5, ls=':', alpha=0.4)
            _label(ax, xlabel='CVaR α', ylabel=label)
            ax.set_xticks(alphas_available)
            ax.set_xticklabels([str(a) for a in alphas_available],
                               color=THEME['subtext'], fontsize=8)

        fig.suptitle('CVaR α-sweep Metrics — QR-DQN + AE encoder · high_vol',
                     color=THEME['text'], fontsize=11)
        fig.tight_layout()
        path = EXP_DIR / 'frontier_metrics_by_alpha.png'
        fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=THEME['bg'])
        plt.close(fig)
        print(f'Saved → {path}')

## 6 — Convergence speed by α

In [ ]:
conv_df = ef.convergence_by_alpha(metric='sharpe')
if not conv_df.empty:
    display(conv_df)
    conv_df.to_csv(EXP_DIR / 'convergence_by_alpha.csv', index=False)

    fastest = conv_df.dropna().sort_values('conv_episode').iloc[0]
    print(f'\nFastest convergence: α={fastest["alpha"]} '
          f'at episode {int(fastest["conv_episode"])}')
else:
    print('No convergence data — run α-sweep first')

## 7 — Full frontier summary

In [ ]:
print(ef.summary())

## 8 — Write-up claim

In [ ]:
if not frontier_df.empty:
    best_sharpe_row = frontier_df.loc[frontier_df.get('sharpe', frontier_df['mean_pnl']).idxmax()]
    best_cvar_row   = frontier_df.loc[frontier_df['cvar_10'].idxmax()]
    alpha_1_row     = frontier_df[frontier_df['alpha'] == 1.0]

    print('CLAIM STATEMENT (efficient frontier):')
    print()
    print(f'The CVaR α-sweep traces a non-dominated efficient frontier in')
    print(f'(mean PnL, CVaR₀.₁₀) space. The risk-neutral baseline (α=1.0)')
    if not alpha_1_row.empty:
        print(f'achieves mean PnL = {float(alpha_1_row["mean_pnl"].iloc[0]):+.4f},')
        print(f'while the most risk-averse variant (α=0.05) achieves')
        alpha_005 = frontier_df[frontier_df['alpha'] == 0.05]
        if not alpha_005.empty:
            print(f'CVaR₀.₁₀ = {float(alpha_005["cvar_10"].iloc[0]):+.4f} '
                  f'at mean PnL = {float(alpha_005["mean_pnl"].iloc[0]):+.4f}.')
    print()
    print('Fill in specific numbers and interpretation after results are available.')